# Cookie Cutter — notebook twin of the anyui UI

Chrome is ipywidgets. Canvases come from `cutter_widgets` (Python twins of `es6/*-cls.js`).

`_esm` is the GitHub Pages file URL so `./curve-editor.js` resolves. Needs network.

In Carnets: save, then **reload from disk** after pulling these files.

`turtlePath` is `[[length, angleDegrees], …]`. `standalone.html` is unchanged.


In [1]:
from __future__ import annotations

import json
import sys
from copy import deepcopy
from pathlib import Path

from IPython.display import display
import ipywidgets as W

root = Path.cwd().resolve()
if not (root / "cutter_widgets" / "__init__.py").is_file():
    for p in root.parents:
        if (p / "cutter_widgets" / "__init__.py").is_file():
            root = p
            break
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from cutter_widgets import CurveEditorWidget, PathTableWidget, WebGLCutterWidget, PAGES_ES6

BLANK = {"name": "Custom", "startPoint": [0.0, 0.0], "startAngle": 0.0, "turtlePath": []}
OUTLINES = {
    "Duck": {
        "name": "Duck",
        "startPoint": [0, 0],
        "startAngle": 180,
        "turtlePath": [
            [0.4, -10], [13.297, 25], [3, -80], [4, 160],
            [22.913, 90], [15, 90], [5, -90], [5, 20],
            [3, 170], [2, -20], [3, -90], [15, 220], [5, -125],
        ],
    },
    "Heart": {
        "name": "Heart",
        "startPoint": [0, 0],
        "startAngle": 180,
        "turtlePath": [
            [0.45, -45], [10, 180], [6.91, -10], [1.1, 110],
            [6.91, -10], [10, 180], [0.45, -45],
        ],
    },
    "Star": {
        "name": "Star",
        "startPoint": [0, 0],
        "startAngle": 0,
        "turtlePath": [[2, -58], [8, 0], [3.2, 130], [8, 0]] * 5,
    },
    "Blank": deepcopy(BLANK),
}
SHAPE_NAMES = ["Duck", "Heart", "Star", "Blank"]

print("root:", root)
print("_esm:", CurveEditorWidget._esm)


root: /private/var/mobile/Containers/Data/Application/96B0A1A4-0205-4CE1-BABA-474BC9B9139A/Documents/Grok/cutter-es6
_esm: https://richardpotthoff.github.io/Grok/cutter-es6/es6/curve-editor-widget.js


In [2]:
from IPython.display import Javascript, display

display(Javascript("""
const url = %s;
alert("pages fetch " + url);
fetch(url, { mode: "cors" })
  .then(r => r.text().then(t => {
    alert("pages status=" + r.status + " type=" + (r.headers.get("content-type") || "") +
          " len=" + t.length + " head=" + t.slice(0, 70));
    return import(url);
  }))
  .then(m => alert("pages import keys=" + Object.keys(m).join(",")))
  .catch(e => alert("pages FAIL " + e));
""" % json.dumps(PAGES_ES6 + "/curve-editor-widget.js")))


<IPython.core.display.Javascript object>

In [3]:
initial = deepcopy(OUTLINES.get("Duck") or next(iter(OUTLINES.values())))
n0 = len(initial["turtlePath"])

editor = CurveEditorWidget(
    name=initial["name"],
    startPoint=list(initial["startPoint"]),
    startAngle=float(initial["startAngle"]),
    turtlePath=[list(s) for s in initial["turtlePath"]],
    selected_index=(n0 - 1) if n0 else -1,
    layout=W.Layout(width="100%", height="320px", min_height="260px", flex="1 1 auto"),
)
viewer = WebGLCutterWidget(
    name=initial["name"],
    startPoint=list(initial["startPoint"]),
    startAngle=float(initial["startAngle"]),
    turtlePath=[list(s) for s in initial["turtlePath"]],
    outlineScale=11.0,
    bladeScale=5.0,
    animate=True,
    layout=W.Layout(width="100%", height="320px", min_height="260px", flex="1 1 auto"),
)
table = PathTableWidget(
    name=initial["name"],
    startPoint=list(initial["startPoint"]),
    startAngle=float(initial["startAngle"]),
    turtlePath=[list(s) for s in initial["turtlePath"]],
    selected_index=editor.selected_index,
    layout=W.Layout(width="100%", max_height="240px"),
)

shape = W.Dropdown(options=SHAPE_NAMES, value="Duck" if "Duck" in SHAPE_NAMES else SHAPE_NAMES[0], description="Shape")
scale = W.FloatText(value=11.0, description="Scale", step=0.5)
btn_fit = W.Button(description="Fit")
btn_insert = W.Button(description="Insert")
btn_delete = W.Button(description="Delete")
btn_export = W.Button(description="Export JSON", button_style="primary")
btn_spin = W.Button(description="Spin")
status = W.HTML(value="<em>Duck loaded — last segment selected. Drag a handle to edit.</em>")

syncing = False


def current_outline() -> dict:
    return editor.get_outline()


def apply_outline(outline: dict, *, keep_selection: bool = True, fit: bool = False) -> None:
    editor.set_outline(outline, keep_selection=keep_selection)
    viewer.set_outline(outline, scale=float(scale.value))
    table.name = editor.name
    table.startPoint = list(editor.startPoint)
    table.startAngle = float(editor.startAngle)
    table.turtlePath = [list(s) for s in editor.turtlePath]
    table.selected_index = editor.selected_index
    btn_delete.disabled = not bool(editor.turtlePath)
    n = len(editor.turtlePath or [])
    status.value = f"<code>{editor.name}</code> · {n} arcs · selected {editor.selected_index}"
    if fit:
        editor.fit()


def with_sync(fn):
    global syncing
    if syncing:
        return
    syncing = True
    try:
        fn()
    finally:
        syncing = False


def push_from_editor(*_):
    def _go():
        outline = current_outline()
        viewer.set_outline(outline, scale=float(scale.value))
        table.name = outline["name"]
        table.startPoint = list(outline["startPoint"])
        table.startAngle = float(outline["startAngle"])
        table.turtlePath = [list(s) for s in outline["turtlePath"]]
        table.selected_index = int(editor.selected_index)
        btn_delete.disabled = not bool(outline["turtlePath"])
        status.value = (
            f"<code>{outline['name']}</code> · {len(outline['turtlePath'])} arcs · "
            f"selected {editor.selected_index}"
        )
    with_sync(_go)


def push_from_table(*_):
    def _go():
        outline = {
            "name": table.name,
            "startPoint": list(table.startPoint),
            "startAngle": float(table.startAngle),
            "turtlePath": [list(s) for s in (table.turtlePath or [])],
        }
        editor.set_outline(outline, keep_selection=True)
        editor.selected_index = int(table.selected_index)
        viewer.set_outline(outline, scale=float(scale.value))
        btn_delete.disabled = not bool(outline["turtlePath"])
    with_sync(_go)


editor.observe(push_from_editor, names=["turtlePath", "startPoint", "startAngle", "name", "selected_index"])
table.observe(push_from_table, names=["turtlePath", "startPoint", "startAngle", "selected_index"])


def on_shape(change):
    name = change["new"]
    outline = deepcopy(OUTLINES.get(name, BLANK))
    if name == "Blank":
        outline = deepcopy(BLANK)
    with_sync(lambda: apply_outline(outline, keep_selection=False, fit=True))


shape.observe(on_shape, names="value")
scale.observe(lambda c: viewer.set_outline(current_outline(), scale=float(c["new"] or 11)), names="value")

btn_fit.on_click(lambda *_: editor.fit())
btn_insert.on_click(lambda *_: editor.insert_segment())
btn_delete.on_click(lambda *_: editor.delete_segment())
btn_spin.on_click(lambda *_: viewer.spin())


def on_export(_=None):
    text = json.dumps(current_outline(), indent=2)
    print(text)
    status.value = f"<pre style='max-height:12rem;overflow:auto'>{text}</pre>"


btn_export.on_click(on_export)

toolbar = W.HBox(
    [shape, scale, btn_fit, btn_insert, btn_delete, btn_export, btn_spin],
    layout=W.Layout(flex_flow="row wrap", align_items="center"),
)
path_panel = W.VBox(
    [W.HTML("<b>Path</b>"), editor],
    layout=W.Layout(width="50%", min_width="16rem", flex="1 1 16rem"),
)
view_panel = W.VBox(
    [W.HTML("<b>3D · Blade</b>"), viewer],
    layout=W.Layout(width="50%", min_width="16rem", flex="1 1 16rem"),
)
stage = W.HBox([path_panel, view_panel], layout=W.Layout(width="100%", align_items="stretch"))
root = W.VBox(
    [
        W.HTML("<h3 style='margin:0 0 8px'>Cookie Cutter</h3>"),
        toolbar,
        stage,
        status,
        table,
    ],
    layout=W.Layout(width="100%"),
)

btn_delete.disabled = not bool(editor.turtlePath)
display(root)


## Files

| Python | JS twin | `_esm` on Pages |
| --- | --- | --- |
| `cutter_widgets/curve_editor.py` | `es6/curve-editor-cls.js` | `curve-editor-widget.js` |
| `cutter_widgets/webgl_cutter.py` | `es6/webgl-cutter-cls.js` | `webgl-cutter-widget.js` |
| `cutter_widgets/path_table.py` | `es6/path-table-cls.js` | `path-table.js` |

Carnets caches the `.ipynb` until you save / reload from disk. After pulling new `.py` files, restart the kernel so `import cutter_widgets` sees them.
